# Solutions · 05 · Temperature dependence and time-temperature superposition

Worked solutions to the exercises of [notebook 05](../notebooks/05_temperature_and_tts.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engrheo is missing): install it from GitHub
    import engrheo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engrheo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engrheo import datasets, tts
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
oil = pd.read_csv(datasets.path("engine_oil_viscosity_temperature.csv"), comment="#")
T, eta = oil.temperature_C.to_numpy() + 273.15, oil.viscosity_Pa_s.to_numpy()

## Exercise 1

Fit the Arrhenius law to the engine-oil data between 60 and 120 degC only, and use it to predict the viscosity at 0 degC. By how much is the cold-start viscosity misjudged?

In [3]:
hot = T >= 333.15
b, a = np.polyfit(1 / T[hot], np.log(eta[hot]), 1)
pred0 = np.exp(a + b / 273.15)
print(f"Arrhenius from 60-120 degC: Ea = {b*tts.R_GAS/1000:.1f} kJ/mol")
print(f"viscosity at 0 degC: predicted {pred0:.3f} Pa s, measured {eta[0]:.3f} Pa s -> predicted/measured = {pred0/eta[0]:.2f}")

Arrhenius from 60-120 degC: Ea = 29.2 kJ/mol
viscosity at 0 degC: predicted 0.331 Pa s, measured 1.011 Pa s -> predicted/measured = 0.33


Extrapolated from the hot range, the Arrhenius law badly underestimates the cold-start viscosity: the
apparent activation energy grows as the oil cools, so the viscosity rises faster than the hot-range line
predicts. Cold-start behaviour must be measured cold - or described with an equation, like Vogel's, that
captures the curvature.

## Exercise 2

With the fitted WLF constants, by what factor does the melt's longest relaxation time change between 170 degC and 220 degC, and between 170 degC and 150 degC? What does that mean for processing?

In [4]:
C1, C2, T_ref = 8.86, 101.6, 443.15
for T_C in (220, 150):
    aT = 10 ** tts.wlf(T_C + 273.15, C1, C2, T_ref)
    print(f"{T_C} degC: relaxation times (and eta0) change by a factor {aT:.3g} relative to 170 degC; longest mode {20*aT:.3g} s")

220 degC: relaxation times (and eta0) change by a factor 0.0012 relative to 170 degC; longest mode 0.0239 s
150 degC: relaxation times (and eta0) change by a factor 148 relative to 170 degC; longest mode 2.97e+03 s


Fifty kelvin hotter, the melt relaxes about a thousand times faster; twenty kelvin colder, about a hundred
times slower. Near the glass transition WLF behaviour is extremely steep: processing temperature is the
strongest lever on flow and relaxation, and a few degrees of temperature non-uniformity in a mould or die
create large differences in orientation and residual stress.

## Exercise 3

**Implement it yourself:** write `wlf_shift(T, C1, C2, T_ref)` and a function that converts WLF constants from one reference temperature to another ($C_2' = C_2 + T_{ref}' - T_{ref}$, $C_1' = C_1C_2/C_2'$); check that both descriptions give identical shift factors.

In [5]:
def wlf_shift(T, C1, C2, T_ref):
    return -C1 * (T - T_ref) / (C2 + T - T_ref)

def wlf_convert(C1, C2, T_ref, T_new):
    C2n = C2 + T_new - T_ref
    return C1 * C2 / C2n, C2n

C1n, C2n = wlf_convert(8.86, 101.6, 443.15, 473.15)
temps = np.array([423.15, 443.15, 473.15, 493.15])
a_old = wlf_shift(temps, 8.86, 101.6, 443.15) - wlf_shift(473.15, 8.86, 101.6, 443.15)   # re-referenced to 473.15 K
print(f"new constants at 200 degC: C1 = {C1n:.3f}, C2 = {C2n:.1f} K")
print("max difference in log10 aT:", np.max(np.abs(a_old - wlf_shift(temps, C1n, C2n, 473.15))))

new constants at 200 degC: C1 = 6.840, C2 = 131.6 K
max difference in log10 aT: 8.881784197001252e-16


The WLF equation has the same form at every reference temperature; changing the reference only changes the
constants. $C_1C_2$ is invariant, which is why tabulated constants must always be quoted with their $T_{ref}$.